Load the pandas library first

In [7]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import coo_matrix
from difflib import get_close_matches

Then load the datasets 

In [8]:
user_history = pd.read_csv('C:/Users/Lwin/Python Deep Learning/User Listening History.csv')
music_info = pd.read_csv('C:/Users/Lwin/Python Deep Learning/Music Info.csv')

encode user and track id

In [11]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

create sparse item matrix

In [12]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

apply truncated SVD

In [13]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)
predicted_ratings = svd_matrix @ svd.components_

convert track name to track id

In [14]:
track_name_to_id = dict(zip(music_info['name'].str.lower(), music_info['track_id']))

recommend based on track name

In [19]:
def recommend_by_track(track_name, top_n=5):
    track_name = track_name.lower()
    if track_name not in track_name_to_id:
        # Suggest similar names
        all_names = music_info['name'].dropna().str.lower().unique()
        suggestions = get_close_matches(track_name, all_names, n=5)
        return f"Track not found. Did you mean: {', '.join(suggestions)}?"

    track_id = track_name_to_id[track_name]
    if track_id not in track_encoder.classes_:
        return "Track found, but not enough data for recommendation."

    track_idx = track_encoder.transform([track_id])[0]

    # Find users who listened to the track
    users_who_listened = user_history[user_history['track_index'] == track_idx]['user_index'].unique()
    if len(users_who_listened) == 0:
        return "No user history found for this track."

    # Average predicted ratings for each item from those users
    avg_scores = predicted_ratings[users_who_listened].mean(axis=0)

    # Remove the original track
    similar_indices = np.argsort(-avg_scores)
    similar_indices = [i for i in similar_indices if i != track_idx][:top_n]

    recommended_ids = track_encoder.inverse_transform(similar_indices)
    return music_info[music_info['track_id'].isin(recommended_ids)][['track_id', 'name', 'artist']].reset_index(drop=True)


Recommend top N songs for track selected

In [28]:
#print(recommend_by_track("love song", top_n=5))  
user_input = input("Enter a track name to get recommendations: ")
results = recommend_by_track(user_input, top_n=5)
print(results)

Enter a track name to get recommendations:  love


             track_id                name             artist
0  TRPFYYL128F92F7144            Float On       MODEST MOUSE
1  TRONYHY128F92C9D11             Revelry      KINGS OF LEON
2  TRXWAZC128F9314B3E  Heartbreak Warfare         JOHN MAYER
3  TRUFTBY128F93450B8           Alejandro          LADY GAGA
4  TRPGPDK12903CCC651    Bring Me To Life  KATHERINE JENKINS
